# Test Performance vs Last Validation From W&B

This notebook follows the same structure as the existing analysis notebook, but it ignores runtime.

It compares:
- final test performance from local `eval_results/`
- the last non-null validation metric logged to W&B during training
- the gap between the two (`test - last_validation`)


In [1]:
import json
import os
from pathlib import Path

import pandas as pd
import wandb


In [2]:
def resolve_repo_root():
    candidates = [Path.cwd(), Path.cwd().resolve(), Path(".").resolve(), Path("..").resolve()]

    try:
        candidates.extend(Path(__file__).resolve().parents)
    except NameError:
        pass

    notebook_hint = Path("notebooks/test_vs_last_val_from_wandb.ipynb").resolve()
    candidates.extend(notebook_hint.parents)

    for candidate in candidates:
        if (candidate / "eval_results").exists() and (candidate / "notebooks").exists():
            return candidate

    raise FileNotFoundError(f"Could not locate repo root from cwd={Path.cwd()}")


ROOT = resolve_repo_root()
EVAL_ROOT = ROOT / "eval_results"
OUTPUT_CSV = ROOT / "notebooks" / "test_vs_last_val_from_wandb.csv"

WANDB_BASE_URL = os.environ.get("WANDB_BASE_URL", "http://localhost:8080")
WANDB_ENTITY = os.environ.get("WANDB_ENTITY", "sebi")
WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "routing_curriculum")

METRIC_KEYS = {
    "arc": "accuracy",
    "sciq": "accuracy",
    "gsm8k": "accuracy_extracted_answer",
}

CURRICULUM_DISPLAY_ORDER = [
    "fixed_k_max",
    "fixed_k_1",
    "linear_k_1_to_topk",
    "frontloaded",
    "backloaded",
    "warmup",
    "jump_warmup",
    "linear_mid_start",
]

BASELINE_CURRICULUM = "fixed_k_max"

ROOT, EVAL_ROOT, OUTPUT_CSV


(PosixPath('/home/sebi/disertatie'),
 PosixPath('/home/sebi/disertatie/eval_results'),
 PosixPath('/home/sebi/disertatie/notebooks/test_vs_last_val_from_wandb.csv'))

## 1. Extract Final Test Metrics

This section only reads local files under `eval_results/`.


In [3]:
def metric_key_for(metrics):
    benchmark = metrics["benchmark"]
    metric_key = METRIC_KEYS.get(benchmark)
    if metric_key is None or metric_key not in metrics:
        raise KeyError(f"Unsupported benchmark metric for {benchmark}: {sorted(metrics.keys())}")
    return metric_key


def parse_run_name(run_name, benchmark):
    if run_name.startswith("base__"):
        return None

    base_run_name, separator, eval_variant = run_name.partition("__")
    if separator and eval_variant != "final":
        return None

    normalized_run_name = base_run_name if separator else run_name
    split_token = f"_{benchmark}_"
    if split_token not in normalized_run_name:
        return None

    model_family, suffix = normalized_run_name.split(split_token, 1)
    curriculum = suffix[:-5] if suffix.endswith("_lora") else suffix
    return model_family, curriculum


EVAL_COLUMNS = [
    "benchmark",
    "model_family",
    "curriculum",
    "run_name",
    "metric_name",
    "test_metric_value",
    "num_examples",
    "checkpoint_path",
    "model_id",
    "metrics_path",
]


def load_eval_table():
    rows = []

    for metrics_path in sorted(EVAL_ROOT.glob("*/*/metrics.json")):
        metrics = json.loads(metrics_path.read_text())
        run_name = metrics_path.parent.name
        benchmark = metrics["benchmark"]
        parsed = parse_run_name(run_name, benchmark)

        if parsed is None:
            continue

        model_family, curriculum = parsed
        metric_key = metric_key_for(metrics)

        rows.append(
            {
                "benchmark": benchmark,
                "model_family": model_family,
                "curriculum": curriculum,
                "run_name": run_name,
                "metric_name": metric_key,
                "test_metric_value": metrics[metric_key],
                "num_examples": metrics.get("num_examples"),
                "checkpoint_path": metrics.get("checkpoint_path"),
                "model_id": metrics.get("model_id"),
                "metrics_path": str(metrics_path),
            }
        )

    eval_df = pd.DataFrame(rows, columns=EVAL_COLUMNS)
    if eval_df.empty:
        return eval_df

    return eval_df.sort_values(["benchmark", "model_family", "curriculum"]).reset_index(drop=True)


In [4]:
eval_df = load_eval_table()
print(f"Eval rows: {len(eval_df)}")
eval_df


Eval rows: 96


,benchmark,model_family,curriculum,run_name,metric_name,test_metric_value,num_examples,checkpoint_path,model_id,metrics_path
0,arc,gpt_oss,backloaded,gpt_oss_arc_backloaded_lora__final,accuracy,0.876280,1172,ckpt/gpt_oss_arc_backloaded_lora/final_adapter,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...
1,arc,gpt_oss,fixed_k_1,gpt_oss_arc_fixed_k_1_lora__final,accuracy,0.846416,1172,ckpt/gpt_oss_arc_fixed_k_1_lora/final_adapter,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...
2,arc,gpt_oss,fixed_k_max,gpt_oss_arc_fixed_k_max_lora__final,accuracy,0.880546,1172,ckpt/gpt_oss_arc_fixed_k_max_lora/final_adapter,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...
3,arc,gpt_oss,frontloaded,gpt_oss_arc_frontloaded_lora__final,accuracy,0.887372,1172,ckpt/gpt_oss_arc_frontloaded_lora/final_adapter,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...
4,arc,gpt_oss,jump_warmup,gpt_oss_arc_jump_warmup_lora__final,accuracy,0.896758,1172,ckpt/gpt_oss_arc_jump_warmup_lora/final_adapter,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...
...,...,...,...,...,...,...,...,...,...,...
91,sciq,qwen,frontloaded,qwen_sciq_frontloaded_lora__final,accuracy,0.926000,1000,ckpt/qwen_sciq_frontloaded_lora/final_adapter,Qwen/Qwen1.5-MoE-A2.7B-Chat,/home/sebi/disertatie/eval_results/sciq/qwen_s...
92,sciq,qwen,jump_warmup,qwen_sciq_jump_warmup_lora__final,accuracy,0.932000,1000,ckpt/qwen_sciq_jump_warmup_lora/final_adapter,Qwen/Qwen1.5-MoE-A2.7B-Chat,/home/sebi/disertatie/eval_results/sciq/qwen_s...
93,sciq,qwen,linear_k_1_to_topk,qwen_sciq_linear_k_1_to_topk_lora__final,accuracy,0.927000,1000,ckpt/qwen_sciq_linear_k_1_to_topk_lora/final_a...,Qwen/Qwen1.5-MoE-A2.7B-Chat,/home/sebi/disertatie/eval_results/sciq/qwen_s...
94,sciq,qwen,linear_mid_start,qwen_sciq_linear_mid_start_lora__final,accuracy,0.928000,1000,ckpt/qwen_sciq_linear_mid_start_lora/final_ada...,Qwen/Qwen1.5-MoE-A2.7B-Chat,/home/sebi/disertatie/eval_results/sciq/qwen_s...


## 2. Query W&B for the Last Validation Metric

For each matched training run, this section reads the task-specific validation metric directly from the W&B run summary.


In [5]:
if WANDB_ENTITY == "REPLACE_ME":
    raise ValueError(
        "Set WANDB_ENTITY before running the W&B cells. Example: export WANDB_ENTITY=your-team-or-username"
    )

api = wandb.Api(overrides={"base_url": WANDB_BASE_URL}, timeout=60)
WANDB_BASE_URL, WANDB_ENTITY, WANDB_PROJECT


wandb: Loading settings from /home/sebi/.config/wandb/settings
wandb: [wandb.Api()] Loaded credentials for http://localhost:8080 from /home/sebi/.netrc.


('http://localhost:8080', 'sebi', 'routing_curriculum')

In [6]:
def load_wandb_last_val_table(eval_df):
    rows = []

    eval_name_lookup = {}
    metric_lookup = {}
    benchmark_lookup = {}

    for row in eval_df.itertuples(index=False):
        eval_name_lookup[row.run_name] = row.run_name
        base_run_name, separator, eval_variant = row.run_name.partition("__")
        if separator and eval_variant == "final":
            eval_name_lookup.setdefault(base_run_name, row.run_name)

        metric_lookup[row.run_name] = f"eval_{row.metric_name}"
        benchmark_lookup[row.run_name] = row.benchmark

    runs = api.runs(f"{WANDB_ENTITY}/{WANDB_PROJECT}")

    for run in runs:
        config = dict(run.config or {})
        summary = dict(run.summary or {})

        output_dir = config.get("output_dir")
        output_run_name = Path(output_dir).name if output_dir else None
        candidate_run_name = output_run_name or run.name

        matched_eval_run_name = eval_name_lookup.get(candidate_run_name) or eval_name_lookup.get(run.name)
        if matched_eval_run_name is None:
            continue

        val_metric_name = metric_lookup[matched_eval_run_name]
        last_val_value = summary.get(val_metric_name)
        if last_val_value is not None:
            last_val_value = float(last_val_value)

        rows.append(
            {
                "run_name": matched_eval_run_name,
                "train_run_name": candidate_run_name,
                "wandb_run_name": run.name,
                "wandb_run_id": run.id,
                "wandb_run_path": "/".join(run.path),
                "wandb_run_url": getattr(run, "url", None),
                "created_at": getattr(run, "created_at", None),
                "state": getattr(run, "state", None),
                "wandb_project": WANDB_PROJECT,
                "dataset_id": config.get("dataset_id"),
                "benchmark": benchmark_lookup[matched_eval_run_name],
                "val_metric_name": val_metric_name,
                "last_val_metric_value": last_val_value,
                "last_val_step": pd.NA,
                "last_val_epoch": pd.NA,
                "best_val_accuracy": summary.get("best_val_accuracy"),
            }
        )

    expected_columns = [
        "run_name",
        "train_run_name",
        "wandb_run_name",
        "wandb_run_id",
        "wandb_run_path",
        "wandb_run_url",
        "created_at",
        "state",
        "wandb_project",
        "dataset_id",
        "benchmark",
        "val_metric_name",
        "last_val_metric_value",
        "last_val_step",
        "last_val_epoch",
        "best_val_accuracy",
    ]

    wandb_df = pd.DataFrame(rows, columns=expected_columns)
    if wandb_df.empty:
        return wandb_df, pd.DataFrame(columns=["run_name", "num_wandb_matches"])

    duplicate_counts = (
        wandb_df.groupby("run_name")
        .size()
        .rename("num_wandb_matches")
        .reset_index()
        .query("num_wandb_matches > 1")
        .sort_values(["num_wandb_matches", "run_name"], ascending=[False, True])
        .reset_index(drop=True)
    )

    wandb_df = wandb_df.sort_values(["created_at", "wandb_run_id"]).drop_duplicates(
        subset="run_name", keep="last"
    )

    return wandb_df.reset_index(drop=True), duplicate_counts


In [7]:
wandb_df, duplicate_wandb_runs = load_wandb_last_val_table(eval_df)
print(f"Matched W&B rows: {len(wandb_df)}")
wandb_df.sort_values(["benchmark", "run_name"]).reset_index(drop=True)


Matched W&B rows: 96


,run_name,train_run_name,wandb_run_name,wandb_run_id,wandb_run_path,wandb_run_url,created_at,state,wandb_project,dataset_id,benchmark,val_metric_name,last_val_metric_value,last_val_step,last_val_epoch,best_val_accuracy
0,gpt_oss_arc_backloaded_lora__final,gpt_oss_arc_backloaded_lora,gpt_oss_arc_backloaded_lora,wyc3wnvc,sebi/routing_curriculum/wyc3wnvc,http://localhost:8080/sebi/routing_curriculum/...,2026-04-19T18:17:37Z,finished,routing_curriculum,None,arc,eval_accuracy,0.903010,<NA>,<NA>,None
1,gpt_oss_arc_fixed_k_1_lora__final,gpt_oss_arc_fixed_k_1_lora,gpt_oss_arc_fixed_k_1_lora,eeal4khf,sebi/routing_curriculum/eeal4khf,http://localhost:8080/sebi/routing_curriculum/...,2026-04-17T09:10:37Z,finished,routing_curriculum,None,arc,eval_accuracy,0.765886,<NA>,<NA>,None
2,gpt_oss_arc_fixed_k_max_lora__final,gpt_oss_arc_fixed_k_max_lora,gpt_oss_arc_fixed_k_max_lora,cwn0oios,sebi/routing_curriculum/cwn0oios,http://localhost:8080/sebi/routing_curriculum/...,2026-04-17T09:02:19Z,finished,routing_curriculum,None,arc,eval_accuracy,0.886288,<NA>,<NA>,None
3,gpt_oss_arc_frontloaded_lora__final,gpt_oss_arc_frontloaded_lora,gpt_oss_arc_frontloaded_lora,hi2k7gtl,sebi/routing_curriculum/hi2k7gtl,http://localhost:8080/sebi/routing_curriculum/...,2026-04-19T18:09:48Z,finished,routing_curriculum,None,arc,eval_accuracy,0.869565,<NA>,<NA>,None
4,gpt_oss_arc_jump_warmup_lora__final,gpt_oss_arc_jump_warmup_lora,gpt_oss_arc_jump_warmup_lora,276gyaym,sebi/routing_curriculum/276gyaym,http://localhost:8080/sebi/routing_curriculum/...,2026-04-19T18:40:29Z,finished,routing_curriculum,None,arc,eval_accuracy,0.896321,<NA>,<NA>,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,qwen_sciq_frontloaded_lora__final,qwen_sciq_frontloaded_lora,qwen_sciq_frontloaded_lora,mk5m1clx,sebi/routing_curriculum/mk5m1clx,http://localhost:8080/sebi/routing_curriculum/...,2026-04-18T18:11:50Z,finished,routing_curriculum,None,sciq,eval_accuracy,0.934000,<NA>,<NA>,None
92,qwen_sciq_jump_warmup_lora__final,qwen_sciq_jump_warmup_lora,qwen_sciq_jump_warmup_lora,45x501yl,sebi/routing_curriculum/45x501yl,http://localhost:8080/sebi/routing_curriculum/...,2026-04-18T20:13:18Z,finished,routing_curriculum,None,sciq,eval_accuracy,0.930000,<NA>,<NA>,None
93,qwen_sciq_linear_k_1_to_topk_lora__final,qwen_sciq_linear_k_1_to_topk_lora,qwen_sciq_linear_k_1_to_topk_lora,lat4zy9p,sebi/routing_curriculum/lat4zy9p,http://localhost:8080/sebi/routing_curriculum/...,2026-04-16T20:46:52Z,finished,routing_curriculum,None,sciq,eval_accuracy,0.945000,<NA>,<NA>,None
94,qwen_sciq_linear_mid_start_lora__final,qwen_sciq_linear_mid_start_lora,qwen_sciq_linear_mid_start_lora,k77qiuso,sebi/routing_curriculum/k77qiuso,http://localhost:8080/sebi/routing_curriculum/...,2026-04-18T19:11:10Z,finished,routing_curriculum,None,sciq,eval_accuracy,0.934000,<NA>,<NA>,None


In [8]:
duplicate_wandb_runs


,run_name,num_wandb_matches


## 3. Join and Compare


In [9]:
comparison_table = eval_df.merge(
    wandb_df.drop(columns=["benchmark"]),
    on="run_name",
    how="left",
    validate="one_to_one",
)

comparison_table["test_minus_last_val"] = (
    comparison_table["test_metric_value"] - comparison_table["last_val_metric_value"]
)

baseline_table = (
    comparison_table.loc[
        comparison_table["curriculum"] == BASELINE_CURRICULUM,
        ["benchmark", "model_family", "test_metric_value", "last_val_metric_value"],
    ]
    .rename(
        columns={
            "test_metric_value": "baseline_test_metric_value",
            "last_val_metric_value": "baseline_last_val_metric_value",
        }
    )
)

comparison_table = comparison_table.merge(
    baseline_table,
    on=["benchmark", "model_family"],
    how="left",
)

comparison_table["test_delta_vs_baseline"] = (
    comparison_table["test_metric_value"] - comparison_table["baseline_test_metric_value"]
)
comparison_table["last_val_delta_vs_baseline"] = (
    comparison_table["last_val_metric_value"] - comparison_table["baseline_last_val_metric_value"]
)

print(f"Rows missing matched W&B validation metric: {comparison_table['last_val_metric_value'].isna().sum()}")
comparison_table.sort_values(["benchmark", "model_family", "curriculum"]).reset_index(drop=True)


Rows missing matched W&B validation metric: 0


,benchmark,model_family,curriculum,run_name,metric_name,test_metric_value,num_examples,checkpoint_path,model_id,metrics_path,...,val_metric_name,last_val_metric_value,last_val_step,last_val_epoch,best_val_accuracy,test_minus_last_val,baseline_test_metric_value,baseline_last_val_metric_value,test_delta_vs_baseline,last_val_delta_vs_baseline
0,arc,gpt_oss,backloaded,gpt_oss_arc_backloaded_lora__final,accuracy,0.876280,1172,ckpt/gpt_oss_arc_backloaded_lora/final_adapter,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...,...,eval_accuracy,0.903010,<NA>,<NA>,None,-0.026730,0.880546,0.886288,-0.004266,0.016722
1,arc,gpt_oss,fixed_k_1,gpt_oss_arc_fixed_k_1_lora__final,accuracy,0.846416,1172,ckpt/gpt_oss_arc_fixed_k_1_lora/final_adapter,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...,...,eval_accuracy,0.765886,<NA>,<NA>,None,0.080530,0.880546,0.886288,-0.034130,-0.120401
2,arc,gpt_oss,fixed_k_max,gpt_oss_arc_fixed_k_max_lora__final,accuracy,0.880546,1172,ckpt/gpt_oss_arc_fixed_k_max_lora/final_adapter,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...,...,eval_accuracy,0.886288,<NA>,<NA>,None,-0.005742,0.880546,0.886288,0.000000,0.000000
3,arc,gpt_oss,frontloaded,gpt_oss_arc_frontloaded_lora__final,accuracy,0.887372,1172,ckpt/gpt_oss_arc_frontloaded_lora/final_adapter,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...,...,eval_accuracy,0.869565,<NA>,<NA>,None,0.017807,0.880546,0.886288,0.006826,-0.016722
4,arc,gpt_oss,jump_warmup,gpt_oss_arc_jump_warmup_lora__final,accuracy,0.896758,1172,ckpt/gpt_oss_arc_jump_warmup_lora/final_adapter,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...,...,eval_accuracy,0.896321,<NA>,<NA>,None,0.000437,0.880546,0.886288,0.016212,0.010033
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,sciq,qwen,frontloaded,qwen_sciq_frontloaded_lora__final,accuracy,0.926000,1000,ckpt/qwen_sciq_frontloaded_lora/final_adapter,Qwen/Qwen1.5-MoE-A2.7B-Chat,/home/sebi/disertatie/eval_results/sciq/qwen_s...,...,eval_accuracy,0.934000,<NA>,<NA>,None,-0.008000,0.929000,0.936000,-0.003000,-0.002000
92,sciq,qwen,jump_warmup,qwen_sciq_jump_warmup_lora__final,accuracy,0.932000,1000,ckpt/qwen_sciq_jump_warmup_lora/final_adapter,Qwen/Qwen1.5-MoE-A2.7B-Chat,/home/sebi/disertatie/eval_results/sciq/qwen_s...,...,eval_accuracy,0.930000,<NA>,<NA>,None,0.002000,0.929000,0.936000,0.003000,-0.006000
93,sciq,qwen,linear_k_1_to_topk,qwen_sciq_linear_k_1_to_topk_lora__final,accuracy,0.927000,1000,ckpt/qwen_sciq_linear_k_1_to_topk_lora/final_a...,Qwen/Qwen1.5-MoE-A2.7B-Chat,/home/sebi/disertatie/eval_results/sciq/qwen_s...,...,eval_accuracy,0.945000,<NA>,<NA>,None,-0.018000,0.929000,0.936000,-0.002000,0.009000
94,sciq,qwen,linear_mid_start,qwen_sciq_linear_mid_start_lora__final,accuracy,0.928000,1000,ckpt/qwen_sciq_linear_mid_start_lora/final_ada...,Qwen/Qwen1.5-MoE-A2.7B-Chat,/home/sebi/disertatie/eval_results/sciq/qwen_s...,...,eval_accuracy,0.934000,<NA>,<NA>,None,-0.006000,0.929000,0.936000,-0.001000,-0.002000


In [10]:
display_table = comparison_table.copy()

for column in [
    "last_val_metric_value",
    "test_metric_value",
    "test_minus_last_val",
    "test_delta_vs_baseline",
    "last_val_delta_vs_baseline",
]:
    display_table[column] = display_table[column].map(
        lambda value: round(value, 4) if pd.notna(value) else value
    )

display_table = display_table.sort_values(["benchmark", "model_family", "curriculum"]).reset_index(drop=True)
display_table.to_csv(OUTPUT_CSV, index=False)
OUTPUT_CSV


PosixPath('/home/sebi/disertatie/notebooks/test_vs_last_val_from_wandb.csv')

In [11]:
display_table[[
    "benchmark",
    "model_family",
    "curriculum",
    "last_val_metric_value",
    "test_metric_value",
    "test_minus_last_val",
    "last_val_step",
    "wandb_run_url",
]]


,benchmark,model_family,curriculum,last_val_metric_value,test_metric_value,test_minus_last_val,last_val_step,wandb_run_url
0,arc,gpt_oss,backloaded,0.9030,0.8763,-0.0267,<NA>,http://localhost:8080/sebi/routing_curriculum/...
1,arc,gpt_oss,fixed_k_1,0.7659,0.8464,0.0805,<NA>,http://localhost:8080/sebi/routing_curriculum/...
2,arc,gpt_oss,fixed_k_max,0.8863,0.8805,-0.0057,<NA>,http://localhost:8080/sebi/routing_curriculum/...
3,arc,gpt_oss,frontloaded,0.8696,0.8874,0.0178,<NA>,http://localhost:8080/sebi/routing_curriculum/...
4,arc,gpt_oss,jump_warmup,0.8963,0.8968,0.0004,<NA>,http://localhost:8080/sebi/routing_curriculum/...
...,...,...,...,...,...,...,...,...
91,sciq,qwen,frontloaded,0.9340,0.9260,-0.0080,<NA>,http://localhost:8080/sebi/routing_curriculum/...
92,sciq,qwen,jump_warmup,0.9300,0.9320,0.0020,<NA>,http://localhost:8080/sebi/routing_curriculum/...
93,sciq,qwen,linear_k_1_to_topk,0.9450,0.9270,-0.0180,<NA>,http://localhost:8080/sebi/routing_curriculum/...
94,sciq,qwen,linear_mid_start,0.9340,0.9280,-0.0060,<NA>,http://localhost:8080/sebi/routing_curriculum/...


## 4. Grouped Tables


In [12]:
curriculum_rank = {name: index for index, name in enumerate(CURRICULUM_DISPLAY_ORDER)}
available_curricula = sorted(
    set(display_table["curriculum"].dropna()),
    key=lambda value: (curriculum_rank.get(value, len(curriculum_rank)), value),
)

CURRICULUM_LABELS = {"fixed_k_max": "Fixed K Default"}


def curriculum_label(curriculum):
    return CURRICULUM_LABELS.get(curriculum, curriculum.replace("_", " ").title())


def format_metric_with_delta(value, delta, is_baseline):
    if pd.isna(value):
        return pd.NA
    if is_baseline or pd.isna(delta):
        return f"{value:.3f}"
    return f"{value:.3f} ({delta:+.3f})"


grouped_table = display_table[[
    "benchmark",
    "model_family",
    "curriculum",
    "last_val_metric_value",
    "test_metric_value",
    "test_minus_last_val",
    "last_val_delta_vs_baseline",
    "test_delta_vs_baseline",
]].copy()

grouped_table["last_val_metric_value"] = grouped_table.apply(
    lambda row: format_metric_with_delta(
        row["last_val_metric_value"],
        row["last_val_delta_vs_baseline"],
        row["curriculum"] == BASELINE_CURRICULUM,
    ),
    axis=1,
)
grouped_table["test_metric_value"] = grouped_table.apply(
    lambda row: format_metric_with_delta(
        row["test_metric_value"],
        row["test_delta_vs_baseline"],
        row["curriculum"] == BASELINE_CURRICULUM,
    ),
    axis=1,
    )
grouped_table["test_minus_last_val"] = grouped_table["test_minus_last_val"].map(
    lambda value: f"{value:+.3f}" if pd.notna(value) else pd.NA
)

last_val_table = grouped_table.pivot(
    index=["benchmark", "model_family"],
    columns="curriculum",
    values="last_val_metric_value",
)
last_val_table = last_val_table.reindex(columns=available_curricula)
last_val_table.columns = [curriculum_label(curriculum) for curriculum in last_val_table.columns]

test_table = grouped_table.pivot(
    index=["benchmark", "model_family"],
    columns="curriculum",
    values="test_metric_value",
)
test_table = test_table.reindex(columns=available_curricula)
test_table.columns = [curriculum_label(curriculum) for curriculum in test_table.columns]

gap_table = grouped_table.pivot(
    index=["benchmark", "model_family"],
    columns="curriculum",
    values="test_minus_last_val",
)
gap_table = gap_table.reindex(columns=available_curricula)
gap_table.columns = [curriculum_label(curriculum) for curriculum in gap_table.columns]

last_val_table


Fixed K Default       Fixed K 1 Linear K 1 To Topk  \
benchmark model_family                                                      
arc       gpt_oss                0.886  0.766 (-0.120)     0.890 (+0.003)   
          lfm2                   0.880  0.271 (-0.609)     0.870 (-0.010)   
          olmoe                  0.689  0.358 (-0.331)     0.679 (-0.010)   
          qwen                   0.829  0.743 (-0.087)     0.819 (-0.010)   
gsm8k     gpt_oss                0.745  0.561 (-0.183)     0.746 (+0.001)   
          lfm2                   0.697  0.381 (-0.316)     0.746 (+0.050)   
          olmoe                  0.499  0.055 (-0.444)     0.495 (-0.004)   
          qwen                   0.638  0.433 (-0.204)     0.624 (-0.013)   
sciq      gpt_oss                0.947  0.901 (-0.046)     0.956 (+0.009)   
          lfm2                   0.948  0.240 (-0.708)     0.942 (-0.006)   
          olmoe                  0.923  0.758 (-0.165)     0.911 (-0.012)   
          qwen                   0.936  0.908 (-0.028)     0.945 (+0.009)   

                           Frontloaded      Backloaded          Warmup  \
benchmark model_family                                                   
arc       gpt_oss       0.870 (-0.017)  0.903 (+0.017)  0.890 (+0.003)   
          lfm2          0.863 (-0.017)  0.863 (-0.017)  0.860 (-0.020)   
          olmoe         0.702 (+0.013)  0.699 (+0.010)  0.676 (-0.013)   
          qwen          0.816 (-0.013)  0.803 (-0.027)  0.836 (+0.007)   
gsm8k     gpt_oss       0.735 (-0.009)  0.734 (-0.011)  0.735 (-0.009)   
          lfm2          0.737 (+0.040)  0.749 (+0.052)  0.725 (+0.028)   
          olmoe         0.497 (-0.001)  0.500 (+0.001)  0.484 (-0.015)   
          qwen          0.631 (-0.007)  0.620 (-0.017)  0.644 (+0.007)   
sciq      gpt_oss       0.950 (+0.003)  0.941 (-0.006)  0.945 (-0.002)   
          lfm2          0.856 (-0.092)  0.538 (-0.410)  0.947 (-0.001)   
          olmoe         0.924 (+0.001)  0.910 (-0.013)  0.925 (+0.002)   
          qwen          0.934 (-0.002)  0.943 (+0.007)  0.942 (+0.006)   

                           Jump Warmup Linear Mid Start  
benchmark model_family                                   
arc       gpt_oss       0.896 (+0.010)   0.896 (+0.010)  
          lfm2          0.866 (-0.013)   0.880 (+0.000)  
          olmoe         0.699 (+0.010)   0.702 (+0.013)  
          qwen          0.819 (-0.010)   0.840 (+0.010)  
gsm8k     gpt_oss       0.738 (-0.007)   0.746 (+0.001)  
          lfm2          0.719 (+0.023)   0.626 (-0.071)  
          olmoe         0.489 (-0.009)   0.484 (-0.015)  
          qwen          0.630 (-0.008)   0.624 (-0.013)  
sciq      gpt_oss       0.955 (+0.008)   0.951 (+0.004)  
          lfm2          0.947 (-0.001)   0.951 (+0.003)  
          olmoe         0.927 (+0.004)   0.921 (-0.002)  
          qwen          0.930 (-0.006)   0.934 (-0.002)

In [13]:
test_table


Fixed K Default       Fixed K 1 Linear K 1 To Topk  \
benchmark model_family                                                      
arc       gpt_oss                0.880  0.846 (-0.034)     0.876 (-0.004)   
          lfm2                   0.842  0.781 (-0.061)     0.838 (-0.004)   
          olmoe                  0.677  0.606 (-0.072)     0.655 (-0.022)   
          qwen                   0.792  0.787 (-0.004)     0.795 (+0.003)   
gsm8k     gpt_oss                0.714  0.559 (-0.155)     0.715 (+0.001)   
          lfm2                   0.598  0.481 (-0.117)     0.634 (+0.036)   
          olmoe                  0.367  0.319 (-0.048)     0.362 (-0.004)   
          qwen                   0.401  0.409 (+0.008)     0.412 (+0.011)   
sciq      gpt_oss                0.951  0.918 (-0.033)     0.954 (+0.003)   
          lfm2                   0.952  0.434 (-0.518)     0.943 (-0.009)   
          olmoe                  0.902  0.884 (-0.018)     0.907 (+0.005)   
          qwen                   0.929  0.926 (-0.003)     0.927 (-0.002)   

                           Frontloaded      Backloaded          Warmup  \
benchmark model_family                                                   
arc       gpt_oss       0.887 (+0.007)  0.876 (-0.004)  0.886 (+0.006)   
          lfm2          0.834 (-0.009)  0.831 (-0.011)  0.836 (-0.006)   
          olmoe         0.677 (+0.000)  0.646 (-0.032)  0.679 (+0.002)   
          qwen          0.792 (+0.000)  0.789 (-0.003)  0.797 (+0.005)   
gsm8k     gpt_oss       0.719 (+0.004)  0.704 (-0.010)  0.718 (+0.004)   
          lfm2          0.640 (+0.042)  0.659 (+0.061)  0.641 (+0.043)   
          olmoe         0.351 (-0.016)  0.352 (-0.014)  0.357 (-0.010)   
          qwen          0.410 (+0.009)  0.409 (+0.008)  0.412 (+0.011)   
sciq      gpt_oss       0.956 (+0.005)  0.951 (+0.000)  0.958 (+0.007)   
          lfm2          0.863 (-0.089)  0.525 (-0.427)  0.947 (-0.005)   
          olmoe         0.903 (+0.001)  0.906 (+0.004)  0.906 (+0.004)   
          qwen          0.926 (-0.003)  0.928 (-0.001)  0.922 (-0.007)   

                           Jump Warmup Linear Mid Start  
benchmark model_family                                   
arc       gpt_oss       0.897 (+0.016)   0.880 (-0.001)  
          lfm2          0.840 (-0.003)   0.840 (-0.002)  
          olmoe         0.679 (+0.002)   0.661 (-0.016)  
          qwen          0.787 (-0.005)   0.787 (-0.004)  
gsm8k     gpt_oss       0.720 (+0.005)   0.720 (+0.006)  
          lfm2          0.616 (+0.017)   0.544 (-0.054)  
          olmoe         0.352 (-0.015)   0.346 (-0.021)  
          qwen          0.401 (+0.000)   0.414 (+0.013)  
sciq      gpt_oss       0.955 (+0.004)   0.950 (-0.001)  
          lfm2          0.955 (+0.003)   0.955 (+0.003)  
          olmoe         0.904 (+0.002)   0.903 (+0.001)  
          qwen          0.932 (+0.003)   0.928 (-0.001)

In [14]:
def highlight_gap_cells(row):
    styles = pd.Series("", index=row.index)

    for column, value in row.items():
        if pd.isna(value):
            continue

        numeric_value = float(str(value).replace("+", ""))
        if numeric_value == 0:
            continue

        color = "#56ad34" if numeric_value > 0 else "#d65f5f"
        styles[column] = f"background-color: {color}; font-weight: 600"

    return styles


(
    gap_table.style.set_caption("Test minus last validation")
    .apply(highlight_gap_cells, axis=1)
    .set_table_styles([
        {"selector": "caption", "props": [("caption-side", "bottom"), ("font-size", "0.95em")]},
        {"selector": "th.col_heading", "props": [("text-align", "center"), ("border-bottom", "1px solid #000")]},
        {"selector": "th.row_heading", "props": [("text-align", "left")]},
    ])
    .set_properties(**{"text-align": "center", "white-space": "nowrap"})
)
